# GuacaMol Benchmarking Testing

This notebook tests the GuacaMol benchmark suite before integrating with the GRASSY model.

GuacaMol provides two types of benchmarks:
1. **Distribution Learning Benchmarks**: Evaluate how well models learn molecular distributions
   - KL divergence of physicochemical properties
   - Fréchet ChemNet Distance (FCD)
   - Novelty, validity, and uniqueness

2. **Goal-Directed Benchmarks**: Optimize specific molecular properties
   - Similarity to target molecules
   - Property optimization (logP, QED, etc.)
   - Multi-objective optimization
   - Rediscovery of known molecules

## References
- GuacaMol paper: https://arxiv.org/abs/1811.09621
- GitHub: https://github.com/BenevolentAI/guacamol

## 1. Install GuacaMol

In [1]:
# Install GuacaMol from PyPI
!pip install guacamol

## 2. Import Libraries

In [8]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, QED
import torch

# GuacaMol imports
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.utils.chemistry import canonicalize

## 3. Load Training Data (ZINC12K)

We'll use ZINC12K as our training distribution.

In [9]:
# Load ZINC12K dataset to extract SMILES
zinc_data = np.load('../datasets/ZINC12K.npy', allow_pickle=True).item()

# The ZINC12K.npy file has SMILES strings as keys and properties as values
# Extract all SMILES strings from the dictionary keys
train_smiles = list(zinc_data.keys())

# Canonicalize SMILES
train_smiles = [canonicalize(s) for s in train_smiles if s is not None]
train_smiles = [s for s in train_smiles if s is not None]  # Remove None values

print(f"Loaded {len(train_smiles)} training SMILES from ZINC12K")
print(f"Example SMILES: {train_smiles[:3]}")

Loaded 11994 training SMILES from ZINC12K
Example SMILES: ['COc1ccc2ccc(O)c(CN3CCN(S(=O)(=O)c4ccn(C)c4)CC3)c2c1', 'CC(C)[NH+](CCNC(=O)N1CCN(C(=O)c2ccncc2)CC1)C1CC1', 'CN(Cc1ccc(Cl)cc1)C(=O)NC1CC1']


## 4. Create a Dummy Generator

GuacaMol requires a generator class that implements the `DistributionMatchingGenerator` interface.
For testing, we'll create a simple dummy generator that samples from the training set.

In [10]:
class DummyGenerator(DistributionMatchingGenerator):
    """
    A simple dummy generator that samples from the training set.
    This is just for testing GuacaMol - replace with your actual model later.
    """
    def __init__(self, training_smiles, mode='random'):
        self.training_smiles = training_smiles
        self.mode = mode
        chemnet_model_filename='ChemNet_v0.13_pretrained.pt'
        
    def generate(self, number_samples: int):
        """
        Generate molecules by sampling from training set.
        
        Args:
            number_samples: Number of molecules to generate
            
        Returns:
            List of SMILES strings
        """
        if self.mode == 'random':
            # Random sampling from training set (with replacement)
            indices = np.random.choice(len(self.training_smiles), size=number_samples, replace=True)
            return [self.training_smiles[i] for i in indices]
        elif self.mode == 'sequential':
            # Sequential sampling (for reproducibility)
            return [self.training_smiles[i % len(self.training_smiles)] for i in range(number_samples)]
        else:
            raise ValueError(f"Unknown mode: {self.mode}")

# Create dummy generator
dummy_gen = DummyGenerator(train_smiles, mode='random')

# Test generation
test_samples = dummy_gen.generate(5)
print(f"\nGenerated {len(test_samples)} test samples:")
for i, smiles in enumerate(test_samples, 1):
    print(f"  {i}. {smiles}")


Generated 5 test samples:
  1. Cc1ccnc(Nc2ncnc(Nc3nc(C)c(C)s3)c2N)c1
  2. CC(C[NH2+]Cc1ccc(Br)s1)Cn1cc[nH+]c1
  3. Cc1cc(Sc2nnc(C3CC3)s2)ccc1[N+](=O)[O-]
  4. Fc1ccc2ncnc(Sc3ncn(-c4ccccc4)n3)c2c1
  5. Cc1noc(C)c1C(C)NC(=O)C1CC1c1cccc(Cl)c1Cl


## 5. Run Distribution Learning Benchmarks

This evaluates how well the generator reproduces the training distribution.

In [11]:
print("Running GuacaMol Distribution Learning Benchmarks...")
print("This will generate 10,000 molecules and compute various metrics.")
print("This may take 5-10 minutes...\n")

# Run the benchmark suite
results = assess_distribution_learning(
    model=dummy_gen,
    chembl_training_file='/Users/joaofelipe/Yale/Grassy/4_Code/GRASSY-Net/notebooks/chemdb_smiles/guacamol_v1_train.smiles',  
    json_output_file='guacamol_results.json',  # Save results
    benchmark_version='v1'  # Use v1 benchmarks
)

print("\n" + "="*70)
print("GUACAMOL DISTRIBUTION LEARNING RESULTS")
print("="*70)

Running GuacaMol Distribution Learning Benchmarks...
This will generate 10,000 molecules and compute various metrics.
This may take 5-10 minutes...



KeyboardInterrupt: 

## 6. Display Benchmark Results

In [ ]:
# Convert results to DataFrame
results_data = []
for result in results:
    results_data.append({
        'Benchmark': result.benchmark_name,
        'Score': result.score
    })

results_df = pd.DataFrame(results_data)
results_df = results_df.sort_values('Score', ascending=False)

print("\n📊 Distribution Learning Benchmark Scores:")
print("="*70)
print(f"{'Benchmark':<40} {'Score':>10}")
print("-"*70)
for _, row in results_df.iterrows():
    print(f"{row['Benchmark']:<40} {row['Score']:>10.4f}")

# Calculate average score
avg_score = results_df['Score'].mean()
print("="*70)
print(f"{'Average Score':<40} {avg_score:>10.4f}")
print("="*70)

# Display as table
print("\nFull Results Table:")
display(results_df)

TypeError: 'NoneType' object is not iterable

## 7. Detailed Metrics Analysis

In [ ]:
# Generate a sample of molecules for detailed analysis
sample_size = 1000
generated_smiles = dummy_gen.generate(sample_size)

print(f"Analyzing {sample_size} generated molecules...\n")

# Validity
valid_mols = [Chem.MolFromSmiles(s) for s in generated_smiles]
valid_count = sum(1 for m in valid_mols if m is not None)
validity = valid_count / len(generated_smiles)

# Uniqueness
unique_smiles = set(generated_smiles)
uniqueness = len(unique_smiles) / len(generated_smiles)

# Novelty (not in training set)
train_set = set(train_smiles)
novel_smiles = unique_smiles - train_set
novelty = len(novel_smiles) / len(unique_smiles) if len(unique_smiles) > 0 else 0

print("📈 Basic Statistics:")
print("-"*50)
print(f"  Validity:    {validity:.4f} ({valid_count}/{len(generated_smiles)})")
print(f"  Uniqueness:  {uniqueness:.4f} ({len(unique_smiles)}/{len(generated_smiles)})")
print(f"  Novelty:     {novelty:.4f} ({len(novel_smiles)}/{len(unique_smiles)})")

# Property distributions
print("\n🔬 Molecular Properties (Valid Molecules):")
print("-"*50)

valid_mols_only = [m for m in valid_mols if m is not None]
if len(valid_mols_only) > 0:
    # Calculate properties
    mol_weights = [Descriptors.MolWt(m) for m in valid_mols_only]
    logps = [Descriptors.MolLogP(m) for m in valid_mols_only]
    qeds = [QED.qed(m) for m in valid_mols_only]
    tpsas = [Descriptors.TPSA(m) for m in valid_mols_only]
    
    properties_df = pd.DataFrame({
        'Molecular Weight': mol_weights,
        'LogP': logps,
        'QED': qeds,
        'TPSA': tpsas
    })
    
    print("\nProperty Statistics:")
    display(properties_df.describe())
else:
    print("  No valid molecules generated!")

## 8. Visualize Property Distributions

In [ ]:
import matplotlib.pyplot as plt

if len(valid_mols_only) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle('Distribution of Molecular Properties', fontsize=16, fontweight='bold')
    
    # Molecular Weight
    axes[0, 0].hist(mol_weights, bins=50, alpha=0.7, color='blue', edgecolor='black')
    axes[0, 0].set_xlabel('Molecular Weight (Da)')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Molecular Weight')
    axes[0, 0].grid(alpha=0.3)
    
    # LogP
    axes[0, 1].hist(logps, bins=50, alpha=0.7, color='green', edgecolor='black')
    axes[0, 1].set_xlabel('LogP')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('LogP (Lipophilicity)')
    axes[0, 1].grid(alpha=0.3)
    
    # QED
    axes[1, 0].hist(qeds, bins=50, alpha=0.7, color='red', edgecolor='black')
    axes[1, 0].set_xlabel('QED Score')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('QED (Drug-likeness)')
    axes[1, 0].grid(alpha=0.3)
    
    # TPSA
    axes[1, 1].hist(tpsas, bins=50, alpha=0.7, color='purple', edgecolor='black')
    axes[1, 1].set_xlabel('TPSA (Ų)')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('TPSA (Polar Surface Area)')
    axes[1, 1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('guacamol_property_distributions.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✅ Property distribution plot saved as 'guacamol_property_distributions.png'")
else:
    print("Cannot create plots - no valid molecules generated")

## 9. Understanding GuacaMol Scores

**Score Interpretation:**
- Scores range from 0.0 to 1.0 (higher is better)
- **≥ 0.9**: Excellent - very close to reference distribution
- **0.7 - 0.9**: Good - reasonable match to distribution
- **0.5 - 0.7**: Moderate - noticeable differences
- **< 0.5**: Poor - significant deviation from reference

**Key Benchmarks:**
1. **validity**: Fraction of chemically valid SMILES
2. **uniqueness**: Fraction of unique molecules
3. **novelty**: Fraction not in training set
4. **KL divergence**: Distribution similarity for various properties
5. **FCD (Fréchet ChemNet Distance)**: Overall distribution similarity
6. **Frechet descriptor distances**: Similarity of molecular descriptors

**Note:** Since our dummy generator just samples from training data:
- Validity and uniqueness should be high
- Novelty will be low (by design)
- KL divergence should be low (good match to training distribution)

## 10. Goal-Directed Benchmarks (Optional)

GuacaMol also provides goal-directed optimization benchmarks.
These require a different generator interface that can optimize for specific targets.
Uncomment to explore these benchmarks.

In [ ]:
# from guacamol.assess_goal_directed_generation import assess_goal_directed_generation
# from guacamol.goal_directed_generator import GoalDirectedGenerator

# class DummyGoalDirectedGenerator(GoalDirectedGenerator):
#     """
#     A dummy goal-directed generator for testing.
#     In practice, this would use optimization to find molecules matching the objective.
#     """
#     def __init__(self, training_smiles):
#         self.training_smiles = training_smiles
#     
#     def generate_optimized_molecules(self, scoring_function, number_molecules: int, 
#                                       starting_population=None):
#         # Dummy implementation: just return random molecules
#         # Your real model would optimize for the scoring_function
#         indices = np.random.choice(len(self.training_smiles), size=number_molecules)
#         return [self.training_smiles[i] for i in indices]

# # Run goal-directed benchmarks
# goal_gen = DummyGoalDirectedGenerator(train_smiles)
# goal_results = assess_goal_directed_generation(
#     goal_gen, 
#     json_output_file='guacamol_goal_directed_results.json'
# )

## 11. Next Steps: Integrate Your Model

Once satisfied with GuacaMol setup, integrate your GRASSY model:

```python
from models.GRASSY_model import GRASSY

class GRASSYGenerator(DistributionMatchingGenerator):
    """
    GuacaMol generator wrapper for GRASSY model.
    """
    def __init__(self, model, device='cpu'):
        self.model = model
        self.device = device
        self.model.eval()
    
    def generate(self, number_samples: int):
        """
        Generate molecules using GRASSY model.
        
        Args:
            number_samples: Number of molecules to generate
            
        Returns:
            List of SMILES strings
        """
        with torch.no_grad():
            # Sample from latent space
            z = torch.randn(number_samples, self.model.latent_dim).to(self.device)
            
            # Decode to molecular representations
            # This depends on your model architecture
            mol_representations = self.model.decode(z)
            
            # Convert to SMILES
            smiles_list = self.convert_to_smiles(mol_representations)
            
        return smiles_list
    
    def convert_to_smiles(self, mol_representations):
        # Implement conversion from your model's output to SMILES
        # This will depend on your specific model architecture
        pass

# Load model and run benchmarks
# model = GRASSY.load_from_checkpoint('path/to/checkpoint.ckpt')
# grassy_gen = GRASSYGenerator(model)
# results = assess_distribution_learning(grassy_gen)
```

In [ ]:
# Placeholder cell for model integration
# Add your GRASSY model code here once GuacaMol testing is complete
pass

## 12. Save Results

In [ ]:
# Results are automatically saved to 'guacamol_results.json'
import json

try:
    with open('guacamol_results.json', 'r') as f:
        saved_results = json.load(f)
    print("✅ Results saved to 'guacamol_results.json'")
    print(f"\nSummary: {len(saved_results['results'])} benchmarks completed")
    print(f"Average score: {saved_results.get('average_score', 'N/A')}")
except FileNotFoundError:
    print("⚠️  Results file not found. Make sure to run the benchmarks first.")